In [1]:
import subprocess
import sys

# -----------------------------
# Install Required Libraries
# -----------------------------

libraries = [
    "numpy",
    "tensorflow",
    "matplotlib",
    "scikit-learn",
    "pillow"
]

print("Installing required libraries...\n")

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"{lib} installed successfully!")
    except Exception as e:
        print(f"Failed to install {lib}: {e}")

print("\nAll required libraries installed successfully!\n")


# -----------------------------
# Imports for Your Code
# -----------------------------

import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import accuracy_score
import numpy as np

print("All libraries imported successfully!")

Installing required libraries...

numpy installed successfully!
tensorflow installed successfully!
matplotlib installed successfully!
scikit-learn installed successfully!
pillow installed successfully!

All required libraries installed successfully!

All libraries imported successfully!


In [2]:
# Define image dimensions and batch size
img_height, img_width = 128, 128
batch_size = 5

# Define directories for training and testing data
train_data_dir = "../dataset/train"
test_data_dir = "../dataset/test"

In [3]:
# Data augmentation for training images
train_datagen = ImageDataGenerator(
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rescale=1./255  # Normalize pixel values
)

# Data augmentation for testing images (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)


In [4]:

# Create data generators for training and testing
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=5,
    class_mode='categorical',
    shuffle=False  # No need to shuffle for evaluation
)

Found 1174 images belonging to 14 classes.
Found 428 images belonging to 14 classes.


In [5]:

# Define the MobileNet base model
base_model = MobileNet(
    input_shape=(img_height, img_width, 3),  # Adjust input shape
    include_top=False,  # Exclude the fully-connected layers
    weights='imagenet'  # Pre-trained on ImageNet
)

# Freeze the base model layers
base_model.trainable = False


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [6]:

# Create the classification head
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(14, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [7]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=15,
    validation_data=test_generator,
    validation_steps=len(test_generator)
)




Epoch 1/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.3612 - loss: 1.9512 - val_accuracy: 0.5958 - val_loss: 1.1626
Epoch 2/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.6337 - loss: 1.1995 - val_accuracy: 0.6636 - val_loss: 1.1198
Epoch 3/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - accuracy: 0.7070 - loss: 0.9497 - val_accuracy: 0.6799 - val_loss: 0.9656
Epoch 4/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 0.7351 - loss: 0.8725 - val_accuracy: 0.7617 - val_loss: 0.7889
Epoch 5/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - accuracy: 0.7862 - loss: 0.7164 - val_accuracy: 0.7593 - val_loss: 0.7634
Epoch 6/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.7922 - loss: 0.6821 - val_accuracy: 0.7593 - val_loss: 0.7684
Epoch 7/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.8032 - loss: 0.6484 - val_accuracy: 0.7360 - val_loss: 0.8921
Epoch 8/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.8186 - loss: 0.5871 - val_a

In [8]:
# predicting train dataset 
test_pridiction = model.predict(test_generator)

# true labels 
test_true_labels = test_generator.classes




86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step


In [9]:
print( test_pridiction.shape)
print( test_true_labels.shape)




(428, 14)
(428,)


In [10]:
#calculating accuracy
accuracy = accuracy_score(test_true_labels, np.argmax(test_pridiction, axis=1))
print(accuracy)

0.8247663551401869


In [11]:
# Save the trained model
model.save("../model_saved_files/Mobilenet.h5")
print("Model saved successfully!")

Model saved successfully!
